# 04 — Evaluation: accuracy with confidence intervals, significance, cost, conformity

- Grades every saved prediction against Bongard's answers (3-vote grader, cached).
- Accuracy with a **95% bootstrap confidence interval** over problems; calls and USD per problem.
- **McNemar tests** on paired problem-level results for the comparisons the paper makes.
- **Conformity:** when a specialist changes its answer in debate, does it move toward the right concept or away?
- **Grader validation sheet** for two people to hand-grade; agreement (Cohen's κ) is computed once filled in.
- Published comparison and figures for the paper.

In [ ]:
%run ./00_common.ipynb

In [ ]:
from scipy.stats import binomtest
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
RUN_GRADER_CHECK = os.environ.get("GRADER_CHECK", "0") == "1"
contrast = load_predictions("contrast")
cv_fail = [bp for bp in PROBLEMS if bp in contrast and not contrast[bp]["verified"]]
design = [bp for bp in PROBLEMS if bp in DEV_DESIGN]
held = [bp for bp in PROBLEMS if bp in DEV_HELDOUT]
held2 = [bp for bp in PROBLEMS if bp in DEV_HELDOUT2]
unseen = [bp for bp in PROBLEMS if bp not in DEV_21]
print(len(PROBLEMS), "problems |", len(cv_fail), "CV proves nothing |", len(design), "used in design |",
      len(held), "held out |", len(held2), "held out (2nd set) |", len(unseen), "never used in design")
print("Design decisions are judged on the held-out columns only; the design column is optimistic by construction.")

## Grader check (optional; labelled pairs incl. reversed relations)

In [ ]:
PAIRS = [
 ("bp0002", "Left = Big shapes, Right = Tiny shapes", True), ("bp0002", "Left = Small figures, Right = Large figures", False),
 ("bp0002", "Left = Outline figures, Right = Solid figures", False),
 ("bp0004", "Left = Shapes with no dents, Right = Shapes that have a concave part", True),
 ("bp0004", "Left = Regular polygons, Right = Irregular shapes", False),
 ("bp0007", "Left = Shapes are vertically oriented, Right = Shapes are horizontally oriented", True),
 ("bp0007", "Left = Figures elongated horizontally, Right = Figures elongated vertically", False),
 ("bp0010", "Left = Three-sided shapes, Right = Four-sided shapes", True), ("bp0010", "Left = Quadrangles, Right = Triangles", False),
 ("bp0031", "Left = A single line, Right = A pair of lines", True),
 ("bp0031", "Left = Single continuous loop, Right = Multiple loops or intersections", False),
 ("bp0036", "Left = The triangle is higher than the circle, Right = The circle is higher than the triangle", True),
 ("bp0036", "Left = Triangles point up, Right = Triangles point down", False),
 ("bp0038", "Left = Triangles are larger than circles, Right = Circles are larger than triangles", True),
 ("bp0038", "Left = Triangle smaller than circle, Right = Triangle larger than circle", False),
 ("bp0047", "Left = A circle surrounds a triangle, Right = A triangle surrounds a circle", True),
 ("bp0047", "Left = Triangle above circle, Right = Circle above triangle", False),
 ("bp0050", "Left = Symmetric figures, Right = Asymmetric figures", True),
 ("bp0050", "Left = Shapes are upright, Right = Shapes are tilted", False),
 ("bp0057", "Left = All figures are exact copies, Right = The figures differ", True),
 ("bp0057", "Left = All shapes are the same size, Right = Shapes are different sizes", False),
 ("bp0036", "Left = triangle above circle, Right = triangle below circle", True),
 ("bp0038", "Left = triangle larger than circle, Right = triangle smaller than circle", True),
 ("bp0038", "Left = circle smaller than triangle, Right = circle larger than triangle", True),
 ("bp0046", "Left = Triangle on top of the circle, Right = Triangle underneath the circle", True),
 ("bp0047", "Left = Circle contains the triangle, Right = Triangle contains the circle", True),
 ("bp0036", "Left = circle below triangle, Right = circle above triangle", True),
 ("bp0036", "Left = triangle below circle, Right = triangle above circle", False),
]
if RUN_GRADER_CHECK:
    METER.tag = "grader check"
    got = [grade_once(bp, rule) for bp, rule, _ in PAIRS]
    print(f"grader agrees with the labels on {np.mean([g == lab for g, (_, _, lab) in zip(got, PAIRS)]):.0%} of {len(PAIRS)} pairs")
    for g, (bp, rule, lab) in zip(got, PAIRS):
        if g != lab:
            print("   disagree:", bp, lab, rule)

## Grade every arm; accuracy with 95% bootstrap CI

In [ ]:
METER.tag = "grading"
rng = np.random.default_rng(0)
def boot_ci(x, n=2000):
    x = np.asarray(x, float)
    if len(x) == 0:
        return (np.nan, np.nan)
    means = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

ours = lambda arm: arm.startswith("pipeline__")
OK, VER, rows = {}, {}, []
for arm in list_arms():
    preds = load_predictions(arm)
    variants = [(arm, "rule")] + ([(arm + " (no check)", "rule_nocheck")] if arm.startswith("pipeline") else [])
    for label, field in variants:
        bps = [bp for bp in PROBLEMS if bp in preds]
        if not bps:
            continue
        ok = {bp: grade(bp, preds[bp].get(field, "")) for bp in bps}
        ver = {bp: bool(preds[bp]["verified"]) if field == "rule" and "verified" in preds[bp]
               else bool(verify_text_rule(preds[bp].get(field, ""), load_cv(bp))) for bp in bps}
        OK[label], VER[label] = ok, ver
        lo, hi = boot_ci(list(ok.values()))
        sub = lambda s: np.mean([ok[b] for b in s if b in ok]) if any(b in ok for b in s) else np.nan
        rows.append({"arm": label, "ours": ours(arm), "n": len(bps), "correct": sum(ok.values()),
                     "accuracy": np.mean(list(ok.values())), "ci_low": lo, "ci_high": hi,
                     "verifier_acc": np.mean(list(ver.values())), "both": np.mean([ok[b] and ver[b] for b in bps]),
                     "acc_cv_fail": sub(cv_fail), "acc_design": sub(design), "acc_heldout": sub(held), "acc_heldout2": sub(held2),
                     "acc_unseen": sub(unseen),
                     "calls_per_problem": float(np.mean([preds[b].get("calls", 0) for b in bps])),
                     "usd_per_problem": float(np.mean([preds[b].get("cost_usd", 0) + preds[b].get("perception_cost_usd", 0) for b in bps]))})
TABLE = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
TABLE["dpl"] = [a / c if c > 0 else np.nan for a, c in zip(TABLE["accuracy"], TABLE["usd_per_problem"])]  # solves per USD
TABLE.to_csv(os.path.join(RESULTS, "tables", "accuracy.csv"), index=False)
pct = "{:.0%}"
print("accuracy = semantic (grader) | verifier_acc = the rule provably separates the panels (CV) | both = both at once")
print("dpl = division-of-perceptual-labour metric: accuracy per USD per problem\n")
display(TABLE.style.format({"accuracy": pct, "ci_low": pct, "ci_high": pct, "acc_cv_fail": pct, "acc_heldout": pct,
                            "acc_design": pct, "acc_heldout2": pct,
                            "verifier_acc": pct, "both": pct, "dpl": "{:.1f}",
                            "acc_unseen": pct, "calls_per_problem": "{:.1f}", "usd_per_problem": "${:.4f}"}))

## McNemar tests (paired, exact) for the comparisons the paper makes

In [ ]:
def mcnemar(a, b, table=None, grader="semantic"):
    """Paired comparison (original stats.paired_binary_summary): risk difference with Wald CI + exact McNemar."""
    T = table or OK
    common = [bp for bp in T.get(a, {}) if bp in T.get(b, {})]
    d = np.array([int(T[a][bp]) - int(T[b][bp]) for bp in common], float)
    b01, b10 = int((d > 0).sum()), int((d < 0).sum())
    p = binomtest(min(b01, b10), b01 + b10, 0.5).pvalue if b01 + b10 else 1.0
    rd = float(d.mean()) if len(d) else 0.0
    half = 1.96 * float(d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else 0.0
    verdict = "A better" if p < 0.05 and rd > 0 else "B better" if p < 0.05 else "no significant difference"
    return {"grader": grader, "A": a, "B": b, "n": len(common), "A right, B wrong": b01, "B right, A wrong": b10,
            "risk difference": rd, "95% CI": f"[{rd - half:+.2f}, {rd + half:+.2f}]", "p (McNemar)": p, "verdict": verdict}

MAIN_ARM = "pipeline__specialist__gpt-oss-120b__ask_eye-judge"
PAIRS_TO_TEST = [(MAIN_ARM, "direct__gpt-4o"),
                 (MAIN_ARM, "pipeline__specialist__gpt-oss-120b__debate-judge"),      # look again vs debate
                 (MAIN_ARM, "pipeline-v6__specialist__gpt-oss-120b"),                  # vs the previous design
                 (MAIN_ARM, "self-consistency__gpt-oss-120b"),
                 (MAIN_ARM, "single__gpt-oss-120b"),
                 (MAIN_ARM, "pipeline__clone__gpt-oss-120b__ask_eye-judge"),
                 (MAIN_ARM, MAIN_ARM + " (no check)"),                                 # judge's rewording vs its pick
                 (MAIN_ARM, "pipeline__specialist__gpt-oss-20b__ask_eye-judge")]
SIG = pd.DataFrame([mcnemar(a, b) for a, b in PAIRS_TO_TEST if a in OK and b in OK] +
                   [mcnemar(a, b, VER, "verifier") for a, b in PAIRS_TO_TEST if a in VER and b in VER])
SIG.to_csv(os.path.join(RESULTS, "tables", "mcnemar.csv"), index=False)
display(SIG)

## Conformity in debate — do agents move toward the right concept, or away?

In [ ]:
CONF_ARMS = [a for a in ("pipeline__specialist__gpt-oss-120b__debate-judge", "pipeline__specialist__gpt-oss-120b__ask_eye-judge",
                          "pipeline__clone__gpt-oss-120b__ask_eye-judge") if load_predictions(a)]
METER.tag = "grading conformity"
conf_rows = []
for arm in CONF_ARMS:
    preds = load_predictions(arm)
    for bp in PROBLEMS:
        r = preds.get(bp)
        if not r or not r.get("rounds"):
            continue
        for agent, h1 in r["round1"].items():
            h2 = r["final_hyps"].get(agent, {})
            if h1.get("rule", "") == h2.get("rule", ""):
                continue
            a, b = grade(bp, h1.get("rule", "")), grade(bp, h2.get("rule", ""))
            conf_rows.append({"arm": arm, "bp": bp, "agent": agent,
                              "kind": "toward correct" if (b and not a) else "away from correct" if (a and not b) else "neutral"})
CONFORMITY = pd.DataFrame(conf_rows)
if len(CONFORMITY):
    display(CONFORMITY.groupby(["arm", "kind"]).size().unstack(fill_value=0))
    CONFORMITY.to_csv(os.path.join(RESULTS, "tables", "conformity.csv"), index=False)
else:
    print("no answer changes in debate on this problem set")

# attempted switches, including the ones the debate guard refused (no panel named as a counterexample)
sw_rows = []
for arm in CONF_ARMS:
    preds = load_predictions(arm)
    for bp in PROBLEMS:
        for sw in (preds.get(bp) or {}).get("switches", []):
            a, b = grade(bp, sw["from"]), grade(bp, sw["to"])
            sw_rows.append({"arm": arm, "bp": bp, "agent": sw["agent"],
                            "outcome": "refused" if sw["kept_own"] else "accepted",
                            "kind": "toward correct" if (b and not a) else "away from correct" if (a and not b) else "neutral"})
SWITCHES = pd.DataFrame(sw_rows)
if len(SWITCHES):
    display(SWITCHES.groupby(["arm", "outcome", "kind"]).size().unstack(fill_value=0))
    SWITCHES.to_csv(os.path.join(RESULTS, "tables", "debate_switches.csv"), index=False)

## Behaviour metrics from the original `behavior_metrics.py` — conformity, volunteering, cognitive islands

In [ ]:
SIM = 0.6
SPECIALTY_KEYWORDS = {
    "shape": ["convex", "concave", "curv", "straight", "triangle", "circle", "square", "polygon", "round", "angular", "symmetr", "outline", "solid"],
    "topology": ["inside", "contain", "touch", "overlap", "connect", "hole", "enclos", "intersect", "cross", "nested"],
    "quantity": ["count", "number", "how many", " one ", " two ", " three ", "more", "fewer", "group", "pair"],
    "spatial": ["position", "top", "bottom", "align", "above", "below", "center", "layout", "vertical", "horizontal", "point"],
    "size": ["large", "small", "big", "tiny", "size", "area", "bigger", "smaller", "wide", "narrow", "tall", "short"],
}
def _sim(a, b):
    return bool(a) and bool(b) and SequenceMatcher(None, a.lower(), b.lower()).ratio() > SIM

def _majority(round_hyps):
    groups = []
    for n, h in round_hyps.items():
        if not h.get("rule"):
            continue
        for g in groups:
            if _sim(h["rule"], round_hyps[g[0]]["rule"]):
                g.append(n); break
        else:
            groups.append([n])
    return max(groups, key=len) if groups else []

def behaviour(r):
    """Conformity: an agent moves INTO the previous round's majority. Volunteering: after debate an agent's rule uses
    another specialty's vocabulary (checked on the rule's content, not the 'Left =/Right =' scaffold, which in the
    original made every rule look spatial). Cognitive island: an agent never in the majority that never yields."""
    r1, r2 = r["round1"], r["final_hyps"]
    maj1 = _majority(r1)
    ref = r1[maj1[0]]["rule"] if maj1 else ""
    conformity = [a for a in r1 if ref and not _sim(r1[a].get("rule", ""), ref) and _sim(r2.get(a, {}).get("rule", ""), ref)]
    volunteer = []
    for a, h in r2.items():
        if a not in SPECIALTY_KEYWORDS or not h.get("rule"):
            continue
        body = " " + " ".join(split_rule(h["rule"])).lower() + " "
        if any(k in body for dim, kws in SPECIALTY_KEYWORDS.items() if dim != a for k in kws):
            volunteer.append(a)
    maj2 = _majority(r2)
    islands = [a for a in r1 if r1[a].get("rule") and a not in maj1 and a not in maj2
               and r1[a].get("rule") == r2.get(a, {}).get("rule")] if r.get("rounds") else []
    return conformity, volunteer, islands

beh_rows = []
for arm in CONF_ARMS:
    preds = load_predictions(arm)
    for bp in PROBLEMS:
        r = preds.get(bp)
        if not r:
            continue
        c, v, isl = behaviour(r)
        beh_rows.append({"arm": arm, "bp": bp, "conformity_events": len(c), "volunteer_events": len(v),
                         "cognitive_island": bool(isl), "debated": bool(r.get("rounds")), "correct": OK.get(arm, {}).get(bp)})
BEH = pd.DataFrame(beh_rows)
if len(BEH):
    display(BEH.groupby("arm").agg(problems=("bp", "count"), debated=("debated", "sum"),
                                   conformity_per_problem=("conformity_events", "mean"),
                                   volunteer_per_problem=("volunteer_events", "mean"),
                                   island_rate=("cognitive_island", "mean")))
    for arm in CONF_ARMS:
        b = BEH[BEH["arm"] == arm]
        if b["cognitive_island"].any() and (~b["cognitive_island"]).any():
            print(f"{arm}: accuracy with a cognitive island {b[b.cognitive_island].correct.mean():.0%} "
                  f"vs without {b[~b.cognitive_island].correct.mean():.0%}")
    BEH.to_csv(os.path.join(RESULTS, "tables", "behaviour.csv"), index=False)
else:
    print("no pipeline predictions on this problem set")

## Grader validation sheet (for two people to hand-grade)

In [ ]:
sheet_path = os.path.join(RESULTS, "tables", "grader_validation_sheet.csv")
if os.path.exists(sheet_path):
    sheet = pd.read_csv(sheet_path)
    filled = sheet.dropna(subset=["human_1", "human_2"]) if {"human_1", "human_2"} <= set(sheet.columns) else sheet.iloc[0:0]
    if len(filled):
        from sklearn.metrics import cohen_kappa_score
        for h in ("human_1", "human_2"):
            k = cohen_kappa_score(filled['grader'], filled[h])
            print(f"grader vs {h}: agreement {np.mean(filled['grader'] == filled[h]):.0%}, Cohen's kappa {k:.2f} "
                  f"(n={len(filled)}) -> {'PASSES' if k >= 0.6 else 'FAILS'} the kappa >= 0.6 gate")
    else:
        print("sheet exists; fill the human_1 / human_2 columns with 1 (same concept) or 0, then re-run")
else:
    items = [(k.split("||")[1], k.split("||")[2], v["match"]) for k, v in _GRADES.items()
             if k.startswith(GRADER_MODEL) and k.split("||")[1] in PROBLEMS]
    random.Random(0).shuffle(items)
    pos = [i for i in items if i[2]][:25]; neg = [i for i in items if not i[2]][:25]
    pd.DataFrame([{"bp": bp, "answer": gt(bp), "rule": rule, "grader": int(m), "human_1": "", "human_2": ""}
                  for bp, rule, m in pos + neg]).sample(frac=1, random_state=0).to_csv(sheet_path, index=False)
    print(f"wrote {sheet_path}: {len(pos) + len(neg)} rows to hand-grade (1 = same concept, 0 = not)")

## Against published results on the classic Bongard problems

In [ ]:
PUBLISHED = pd.DataFrame([
    {"system": "o1", "solved of 100": 43, "note": "reported on the original 100; their own protocol"},
    {"system": "Claude 3.5 Sonnet", "solved of 100": 31, "note": "reported on the original 100; their own protocol"},
    {"system": "GPT-4o", "solved of 100": 24, "note": "reported on the original 100; their own protocol"},
    {"system": "Phaeaco (Foundalis)", "solved of 100": 10, "note": "dedicated cognitive architecture"},
])
display(PUBLISHED)
if set(PROBLEMS) != set(ALL_100):
    print(f"NOTE: {len(PROBLEMS)} problems, not all 100 — a development signal, not comparable to the published numbers.")

## Figures

In [ ]:
from matplotlib.patches import Patch
def arm_label(a):
    return a.replace("pipeline__", "pipeline · ").replace("__", " · ")
show = TABLE[~TABLE["arm"].str.startswith("pipeline-v1")].iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(show) + 1.4))
ys = range(len(show))
ax.barh([arm_label(a) for a in show["arm"]], show["accuracy"] * 100,
        color=[C_OURS if o else C_BASE for o in show["ours"]], height=0.6)
ax.errorbar(show["accuracy"] * 100, list(ys),
            xerr=[(show["accuracy"] - show["ci_low"]) * 100, (show["ci_high"] - show["accuracy"]) * 100],
            fmt="none", ecolor=C_INK2, elinewidth=1, capsize=3)
for y, (acc, n, k, hi) in enumerate(zip(show["accuracy"], show["n"], show["correct"], show["ci_high"])):
    ax.text(hi * 100 + 1.5, y, f"{acc:.0%} ({k}/{n})", va="center", color=C_INK2, fontsize=9)
ax.set_xlim(0, 110); ax.set_xlabel("problems solved (%) with 95% bootstrap CI")
ax.legend(handles=[Patch(color=C_OURS, label="our pipeline"), Patch(color=C_BASE, label="baselines")], loc="lower right", frameon=False)
ax.set_title(f"Accuracy on {len(PROBLEMS)} Bongard problems", loc="left")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "figures", "accuracy_by_arm.png"), dpi=200); plt.show()

main_arms = show[~show["arm"].str.contains("no check")]
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for _, r in main_arms.iterrows():
    x = max(r["usd_per_problem"], 1e-4)
    ax.scatter(x, r["accuracy"] * 100, s=70, color=C_OURS if r["ours"] else C_BASE, edgecolor=C_SURFACE, linewidth=2, zorder=3)
    ax.annotate(arm_label(r["arm"]), (x, r["accuracy"] * 100), textcoords="offset points", xytext=(7, 4), fontsize=8, color=C_INK2)
ax.set_xscale("log"); ax.set_xlabel("cost per problem (USD, log scale)"); ax.set_ylabel("problems solved (%)"); ax.set_ylim(0, 100)
ax.legend(handles=[Patch(color=C_OURS, label="our pipeline"), Patch(color=C_BASE, label="baselines")], loc="upper left", frameon=False)
ax.set_title("Accuracy against cost", loc="left")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, "figures", "accuracy_vs_cost.png"), dpi=200); plt.show()
METER.report()